[View the live Snake Game here!](https://akhilkulkarni123.github.io/TheSprinters/snake1)

# Snake Game Lesson

## What You'll Learn
This lesson walks through a complete Snake game built with HTML, CSS, and JavaScript. It's a fun monkey and banana themed game that teaches you the basics of web game development!


## 1. Building the Game Structure with HTML

### Creating Different Screens
Our game has 4 different screens that the player can see. Think of them like different pages in a book - only one is visible at a time.

```html
<!-- Main Menu Screen - what players see first -->
<div id="menu" class="py-4 text-light">
    <p>Welcome to Monkey Snake, press <span style="background-color: #FFFFFF; color: #000000">space</span> to begin</p>
    <p>Use <span style="background-color: #FFFFFF; color: #000000">arrow keys</span> or <span style="background-color: #FFFFFF; color: #000000">WASD</span> to move</p>
    <p style="color: #ff6b6b;">⚠️ Deadly obstacles appear at score 3!</p>
    <a id="new_game" class="link-alert">new game</a>
    <a id="setting_menu" class="link-alert">settings</a>
</div>

<!-- Game Over Screen - shows when player loses -->
<div id="gameover" class="py-4 text-light">
    <p>Game Over, press <span style="background-color: #FFFFFF; color: #000000">space</span> to try again</p>
    <p>Use <span style="background-color: #FFFFFF; color: #000000">arrow keys</span> or <span style="background-color: #FFFFFF; color: #000000">WASD</span> to move</p>
    <a id="new_game1" class="link-alert">new game</a>
    <a id="setting_menu1" class="link-alert">settings</a>
</div>

<!-- The Game Canvas - where the actual game happens -->
<canvas id="snake" class="wrap" width="320" height="320" tabindex="1"></canvas>

<!-- Settings Screen - where players can change game options -->
<div id="setting" class="py-4 text-light">
    <p>Settings Screen, press <span style="background-color: #FFFFFF; color: #000000">space</span> to go back to playing</p>
    <a id="new_game2" class="link-alert">new game</a>
    <br>
    <p>Speed:
        <input id="speed1" type="radio" name="speed" value="120" checked/>
        <label for="speed1">Slow</label>
        <input id="speed2" type="radio" name="speed" value="75"/>
        <label for="speed2">Normal</label>
        <input id="speed3" type="radio" name="speed" value="35"/>
        <label for="speed3">Fast</label>
    </p>
    <p>Wall:
        <input id="wallon" type="radio" name="wall" value="1" checked/>
        <label for="wallon">On</label>
        <input id="walloff" type="radio" name="wall" value="0"/>
        <label for="walloff">Off</label>
    </p>
</div>
```

**What's Happening Here:**
- **Different screens**: Each `<div>` is a different screen the player can see
- **Canvas element**: The `<canvas>` is like a blank piece of paper where we'll draw our game
- **Size matters**: 320x320 pixels gives us a square game area
- **Radio buttons**: Let players choose between different options (like fast vs slow speed)
- **IDs**: Each element has a unique name so JavaScript can find and control it

### Why Use Canvas?
The HTML5 Canvas is perfect for games because:
- We can draw shapes, images, and text anywhere we want
- We can update the screen many times per second to create animation
- We can detect mouse and keyboard input easily

---

## 2. Making It Look Cool with CSS

### Glowing Border Effect
```css
canvas {
    display: none;                    /* Hide the game until it starts */
    border-style: solid;              /* Solid border around the game area */
    border-width: 10px;               /* Thick border so players can see the edges */
    border-color: #FF0000;            /* Red color for intensity */
    box-shadow: 0 0 20px #FF0000,     /* Inner glow */
                0 0 40px #FF0000,     /* Medium glow */
                0 0 60px #FF0000;     /* Outer glow */
    animation: borderGlow 2s ease-in-out infinite alternate;
}

@keyframes borderGlow {
    from {
        box-shadow: 0 0 20px #FF0000, 0 0 40px #FF0000, 0 0 60px #FF0000;
    }
    to {
        box-shadow: 0 0 30px #FF6666, 0 0 60px #FF6666, 0 0 90px #FF6666;
    }
}
```

**What This Does:**
- **display: none**: Keeps the canvas hidden until the game starts
- **border**: Creates a thick red border around the game
- **box-shadow**: Makes the border glow with multiple shadow layers
- **animation**: Makes the glow pulse from dark red to light red every 2 seconds
- **infinite alternate**: The animation goes back and forth forever

### Cool Hover Effects for Menu Items
```css
#menu a:hover::before {
    content: ">";                     /* Adds a ">" symbol */
    margin-right: 10px;               /* Spaces it away from the text */
}
```

**What This Does:**
- **:hover**: Only happens when the mouse is over the menu item
- **::before**: Adds content before the text
- **content: ">"**: Puts a little arrow to show which item you're hovering over

### Custom Radio Buttons
```css
#setting input {
    display: none;                    /* Hide the ugly default radio buttons */
}

#setting input:checked + label {
    background-color: #FFF;           /* White background for selected option */
    color: #000;                      /* Black text for contrast */
}
```

**What This Does:**
- **display: none**: Hides the default radio buttons (they're ugly!)
- **:checked + label**: When a radio button is selected, style the label next to it
- **background-color**: Makes selected options have a white background so you can see what's picked

---

## 3. JavaScript: The Brain of the Game

### Setting Up Our Variables
```javascript
// Get the canvas and drawing context
const canvas = document.getElementById("snake");
const ctx = canvas.getContext("2d");

// Screen constants - numbers that represent different screens
const SCREEN_SNAKE = 0;               // Playing the game
const SCREEN_MENU = -1;               // Main menu
const SCREEN_GAME_OVER = 1;           // Game over screen
const SCREEN_SETTING = 2;             // Settings screen

// Game variables
const BLOCK = 10;                     // Each grid square is 10 pixels
let SCREEN = SCREEN_MENU;             // Start at the main menu
let snake;                            // Array to hold all snake pieces
let snake_dir;                        // Which direction snake is moving (0-3)
let snake_next_dir;                   // Next direction (for smooth controls)
let snake_speed;                      // How fast the game runs
let food = {x: 0, y: 0};             // Where the banana is
let score;                            // Player's current score
let wall;                             // Whether walls kill you or not
let obstacles = [];                   // Array of rocks to avoid
```

**What These Variables Do:**
- **const**: Variables that never change (like BLOCK size)
- **let**: Variables that will change during the game
- **Arrays**: Lists of things (like snake pieces and obstacles)
- **Objects**: Things with properties (like food having x and y coordinates)

### Background Colors for Visual Fun
```javascript
// Different background colors that change as you play
let backgroundColors = [
    "royalblue",      // Cool blue
    "purple",         // Rich purple
    "darkgreen",      // Forest green
    "maroon",         // Deep red
    "darkslategray",  // Gray
    "midnightblue",   // Dark blue
    "indigo",         // Purple-blue
    "darkred",        // Dark red
    "forestgreen",    // Green
    "darkorange"      // Orange
];
let currentColorIndex = 0;            // Which color we're currently using

// Background image (if it loads)
let backgroundImage = new Image();
backgroundImage.src = 'images/junglebg.png';
```

**Why This is Cool:**
- Every time you eat food, the background color changes
- 10 different colors keep the game visually interesting
- If the background image doesn't load, we still have colors as backup

### Switching Between Screens
```javascript
let showScreen = function(screen_opt){
    SCREEN = screen_opt;              // Remember which screen we're on
    switch(screen_opt){
        case SCREEN_SNAKE:            // Show the game
            screen_snake.style.display = "block";     // Show canvas
            screen_menu.style.display = "none";       // Hide menu
            screen_setting.style.display = "none";    // Hide settings
            screen_game_over.style.display = "none";  // Hide game over
            break;
        case SCREEN_GAME_OVER:        // Show game over
            screen_snake.style.display = "block";     // Keep canvas visible
            screen_menu.style.display = "none";       // Hide menu
            screen_setting.style.display = "none";    // Hide settings
            screen_game_over.style.display = "block"; // Show game over text
            break;
        case SCREEN_SETTING:          // Show settings
            screen_snake.style.display = "none";      // Hide canvas
            screen_menu.style.display = "none";       // Hide menu
            screen_setting.style.display = "block";   // Show settings
            screen_game_over.style.display = "none";  // Hide game over
            break;
    }
}
```

**How This Works:**
- Only one screen shows at a time
- We hide all screens, then show just the one we want
- **switch** statement is like a bunch of if-else statements but cleaner

---

## 4. The Game Loop: Making Things Move

### The Main Game Loop
```javascript
let mainLoop = function(){
    // 1. Figure out where the snake's head should move
    let _x = snake[0].x;              // Current head x position
    let _y = snake[0].y;              // Current head y position
    snake_dir = snake_next_dir;       // Update direction
    
    // 2. Move based on direction (0=Up, 1=Right, 2=Down, 3=Left)
    switch(snake_dir){
        case 0: _y--; break;          // Up means y gets smaller
        case 1: _x++; break;          // Right means x gets bigger
        case 2: _y++; break;          // Down means y gets bigger
        case 3: _x--; break;          // Left means x gets smaller
    }
    
    // 3. Move the snake: remove tail, add new head
    snake.pop();                      // Remove the last piece (tail)
    snake.unshift({x: _x, y: _y});    // Add new piece at front (head)
    
    // 4. Check if snake hit a wall
    if(wall === 1){
        // If walls are ON, hitting them ends the game
        if (snake[0].x < 0 || snake[0].x === canvas.width / BLOCK || 
            snake[0].y < 0 || snake[0].y === canvas.height / BLOCK){
            showScreen(SCREEN_GAME_OVER);
            return;                   // Stop the game loop
        }
    } else {
        // If walls are OFF, snake wraps around to other side
        for(let i = 0; i < snake.length; i++){
            if(snake[i].x < 0){
                snake[i].x = snake[i].x + (canvas.width / BLOCK);
            }
            if(snake[i].x === canvas.width / BLOCK){
                snake[i].x = snake[i].x - (canvas.width / BLOCK);
            }
            if(snake[i].y < 0){
                snake[i].y = snake[i].y + (canvas.height / BLOCK);
            }
            if(snake[i].y === canvas.height / BLOCK){
                snake[i].y = snake[i].y - (canvas.height / BLOCK);
            }
        }
    }
    
    // 5. Check if snake ran into itself
    for(let i = 1; i < snake.length; i++){
        if (snake[0].x === snake[i].x && snake[0].y === snake[i].y){
            showScreen(SCREEN_GAME_OVER);
            return;                   // Game over!
        }
    }
    
    // 6. Check if snake hit an obstacle (rock)
    for(let i = 0; i < obstacles.length; i++){
        if(checkBlock(snake[0].x, snake[0].y, obstacles[i].x, obstacles[i].y)){
            showScreen(SCREEN_GAME_OVER);
            return;                   // Game over!
        }
    }
    
    // 7. Check if snake ate food
    if(checkBlock(snake[0].x, snake[0].y, food.x, food.y)){
        // Snake grows by adding a new piece
        snake[snake.length] = {x: snake[0].x, y: snake[0].y};
        
        // Increase score
        altScore(++score);
        
        // Change background color for fun
        currentColorIndex = (currentColorIndex + 1) % backgroundColors.length;
        
        // Make new food appear
        addFood();
        
        // Add obstacles to make game harder
        if(score >= 3 && (score - 3) % 2 === 0){
            addObstacle();
        }
    }
    
    // 8. Draw everything on the screen
    ctx.beginPath();
    
    // Draw background
    if (backgroundImage.complete && backgroundImage.naturalHeight !== 0) {
        // Mix color with background image
        ctx.globalAlpha = 0.7;
        ctx.fillStyle = backgroundColors[currentColorIndex];
        ctx.fillRect(0, 0, canvas.width, canvas.height);
        ctx.globalAlpha = 0.6;
        ctx.drawImage(backgroundImage, 0, 0, canvas.width, canvas.height);
        ctx.globalAlpha = 1.0;        // Reset transparency
    } else {
        // Just use solid color if image didn't load
        ctx.fillStyle = backgroundColors[currentColorIndex];
        ctx.fillRect(0, 0, canvas.width, canvas.height);
    }
    
    // Draw all snake pieces
    for(let i = 0; i < snake.length; i++){
        drawMonkey(snake[i].x, snake[i].y);
    }
    
    // Draw food
    drawBanana(food.x, food.y);
    
    // Draw all obstacles
    for(let i = 0; i < obstacles.length; i++){
        drawObstacle(obstacles[i].x, obstacles[i].y);
    }
    
    // 9. Wait a bit, then do this all over again
    setTimeout(mainLoop, snake_speed);  // This makes the game loop repeat
}
```

**How the Game Loop Works:**
- This function runs over and over again (like 10 times per second)
- Each time it runs, it moves the snake a little bit
- It checks if anything bad happened (hitting walls, itself, or rocks)
- It checks if anything good happened (eating food)
- It draws everything on the screen
- Then it waits a little bit and does it all again

### Simple Collision Detection
```javascript
let checkBlock = function(x, y, _x, _y){
    return (x === _x && y === _y);    // Are these two positions exactly the same?
}
```

**Why This Works:**
- Our game uses a grid system (like graph paper)
- Two things collide if they're in the exact same grid square
- This function just checks if the x AND y coordinates are the same

---

## 5. Handling Player Input

### Keyboard Controls
```javascript
let changeDir = function(key){
    // Don't let the snake reverse direction instantly (that would be cheating!)
    switch(key) {
        case 37:    // Left arrow key
        case 65:    // A key
            if (snake_dir !== 1)        // Only if not moving right
                snake_next_dir = 3;     // Change to left
            break;
        case 38:    // Up arrow key
        case 87:    // W key
            if (snake_dir !== 2)        // Only if not moving down
                snake_next_dir = 0;     // Change to up
            break;
        case 39:    // Right arrow key
        case 68:    // D key
            if (snake_dir !== 3)        // Only if not moving left
                snake_next_dir = 1;     // Change to right
            break;
        case 40:    // Down arrow key
        case 83:    // S key
            if (snake_dir !== 0)        // Only if not moving up
                snake_next_dir = 2;     // Change to down
            break;
    }
}

// Listen for key presses when canvas is focused
canvas.onkeydown = function(evt) {
    changeDir(evt.keyCode);             // Convert key press to direction change
}
```

**Smart Direction Changing:**
- Players can use arrow keys OR WASD (like in many PC games)
- Snake can't reverse direction instantly (that would make the game too easy)
- We use numbers for directions: 0=Up, 1=Right, 2=Down, 3=Left

### Global Controls
```javascript
// Listen for spacebar on the whole page
window.addEventListener("keydown", function(evt) {
    if(evt.code === "Space" && SCREEN !== SCREEN_SNAKE)
        newGame();                      // Start new game if not currently playing
}, true);
```

**Why This is Useful:**
- Spacebar starts a new game from any screen except when already playing
- Works no matter where the player clicked last

---

## 6. Drawing Graphics

### Using Emojis as Game Graphics
```javascript
let drawMonkey = function(x, y){
    ctx.font = "10px Arial";           // Set font size to match our grid
    ctx.fillText("🐵", x * BLOCK, (y * BLOCK) + BLOCK);
}

let drawBanana = function(x, y){
    ctx.font = "10px Arial";           // Same size for consistency
    ctx.fillText("🍌", x * BLOCK, (y * BLOCK) + BLOCK);
}

let drawObstacle = function(x, y){
    ctx.font = "10px Arial";           // Same size for all game objects
    ctx.fillText("🪨", x * BLOCK, (y * BLOCK) + BLOCK);
}
```

**Why Use Emojis:**
- They're colorful and fun to look at
- No need to create or download image files
- They work on all computers and phones
- Perfect for a beginner project

### Converting Grid to Pixels
```javascript
x * BLOCK      // Convert grid position to pixel position
y * BLOCK      // Same for Y coordinate
(y * BLOCK) + BLOCK  // Y position adjusted for how text is drawn
```

**Understanding Coordinates:**
- Our game logic thinks in grid squares (0 to 31)
- Canvas drawing needs pixel coordinates (0 to 320)
- Multiply by BLOCK (10) to convert grid to pixels

---

## 7. Making the Game Challenging

### Adding Obstacles as Score Increases
```javascript
if(score >= 3 && (score - 3) % 2 === 0){
    addObstacle();
}
```

**How This Works:**
- First obstacle appears when score reaches 3
- After that, new obstacle every 2 points (at scores 5, 7, 9, etc.)
- Gives players time to learn before making it harder

### Smart Obstacle Placement
```javascript
let addObstacle = function(){
    let obstacle = {
        x: Math.floor(Math.random() * ((canvas.width / BLOCK) - 1)),
        y: Math.floor(Math.random() * ((canvas.height / BLOCK) - 1))
    };
    
    // Make sure obstacle doesn't appear on snake or food
    let validPosition = false;
    let attempts = 0;
    while (!validPosition && attempts < 50) {
        validPosition = true;
        
        // Check if it would hit any snake piece
        for(let i = 0; i < snake.length; i++){
            if(checkBlock(obstacle.x, obstacle.y, snake[i].x, snake[i].y)){
                validPosition = false;
                break;                  // Stop checking, we found a problem
            }
        }
        
        // Check if it would hit the food
        if(checkBlock(obstacle.x, obstacle.y, food.x, food.y)){
            validPosition = false;
        }
        
        // Check if it would hit other obstacles
        for(let i = 0; i < obstacles.length; i++){
            if(checkBlock(obstacle.x, obstacle.y, obstacles[i].x, obstacles[i].y)){
                validPosition = false;
                break;                  // Stop checking, we found a problem
            }
        }
        
        // If there's a problem, try a new random position
        if(!validPosition){
            obstacle.x = Math.floor(Math.random() * ((canvas.width / BLOCK) - 1));
            obstacle.y = Math.floor(Math.random() * ((canvas.height / BLOCK) - 1));
        }
        attempts++;                     // Count how many times we've tried
    }
    
    // Only add the obstacle if we found a good spot
    if(validPosition){
        obstacles.push(obstacle);
    }
}
```

**Why This is Important:**
- Obstacles shouldn't appear on top of the snake (that would be unfair!)
- They shouldn't appear on food (that would be confusing)
- If we can't find a good spot after 50 tries, we just skip adding one

### Smart Food Placement
```javascript
let addFood = function(){
    // Pick a random spot
    food.x = Math.floor(Math.random() * ((canvas.width / BLOCK) - 1));
    food.y = Math.floor(Math.random() * ((canvas.height / BLOCK) - 1));
    
    // If food appears on snake, try again
    for(let i = 0; i < snake.length; i++){
        if(checkBlock(food.x, food.y, snake[i].x, snake[i].y)){
            addFood();                  // Call this function again (recursion!)
            return;
        }
    }
    
    // If food appears on obstacle, try again
    for(let i = 0; i < obstacles.length; i++){
        if(checkBlock(food.x, food.y, obstacles[i].x, obstacles[i].y)){
            addFood();                  // Call this function again
            return;
        }
    }
}
```

**Recursion Explained:**
- If food appears in a bad spot, the function calls itself to try again
- This keeps happening until food appears in a good spot
- It's like saying "keep rolling the dice until you get what you want"

---

## 8. Game Settings and Customization

### Speed Control
```javascript
let setSnakeSpeed = function(speed_value){
    snake_speed = speed_value;        // How many milliseconds between game updates
}

// Speed options:
// 120 = slow (waits 120 milliseconds between moves)
// 75 = normal (waits 75 milliseconds between moves)
// 35 = fast (waits 35 milliseconds between moves)
```

**How Speed Works:**
- Lower numbers = faster game (less waiting between moves)
- Higher numbers = slower game (more waiting between moves)
- Players can change this in the settings screen

### Wall Settings
```javascript
let setWall = function(wall_value){
    wall = wall_value;
    if(wall === 0){screen_snake.style.borderColor = "#FF0000";}  // Walls off
    if(wall === 1){screen_snake.style.borderColor = "#FF0000";}  // Walls on
}
```

**Two Different Game Modes:**
- **Walls ON**: Hitting the edge ends the game (harder)
- **Walls OFF**: Snake wraps around to the other side (easier)

---

## 9. Starting a New Game

### Setting Up a Fresh Game
```javascript
let newGame = function(){
    // Switch to the game screen
    showScreen(SCREEN_SNAKE);
    screen_snake.focus();
    
    // Reset score to zero
    score = 0;
    altScore(score);
    
    // Create a new snake with just one piece
    snake = [];
    snake.push({x: 0, y: 15});
    snake_next_dir = 1;               // Start moving right
    
    // Remove all obstacles from previous game
    obstacles = [];
    
    // Reset background color
    currentColorIndex = 0;
    
    // Put food somewhere on the screen
    addFood();
    
    // Start listening for arrow key presses
    canvas.onkeydown = function(evt) {
        changeDir(evt.keyCode);
    }
    
    // Start the game loop!
    mainLoop();
}
```

**What Happens When Starting a New Game:**
- Switch to the game screen
- Reset everything to starting values
- Create a tiny snake with just one piece
- Clear out any obstacles from the last game
- Put food on the screen
- Start listening for player input
- Begin the main game loop

---

## 10. Why This Code Structure Works

### Breaking Things Into Functions
Each function has one job:
- `drawMonkey()` - just draws a monkey emoji
- `checkBlock()` - just checks if two things are in the same spot
- `addFood()` - just puts food in a good location
- `mainLoop()` - coordinates everything else

**Why This is Good:**
- Easy to understand what each piece does
- Easy to fix bugs (you know exactly where to look)
- Easy to add new features
- Other programmers can understand your code

### Using Meaningful Variable Names
- `snake_speed` instead of just `s`
- `currentColorIndex` instead of just `i`
- `validPosition` instead of just `ok`

**Why This Matters:**
- You can read the code like English
- When you come back to your code months later, you'll still understand it
- Other people can help you with your code

### Game Loop Pattern
Almost every game uses this same pattern:
1. Get player input
2. Update game objects (move things)
3. Check for collisions
4. Update score/game state
5. Draw everything
6. Wait a little bit
7. Repeat

**This is Professional:**
- Real game developers use this exact same pattern
- Once you understand it, you can make any kind of game
- It keeps everything organized and predictable

This game shows you all the basics of game programming in a fun, understandable way. You can use these same concepts to build much more complex games!